# Tujuan

Notebook ini bertanggung jawab untuk:

1. Preparasi & Verifikasi File Sumber
    - Memvalidasi source file (existence, tipe, ukuran).
    - Mengambil fingerprint source menggunakan SHA-256.
    - Memvalidasi struktur CSV mentah (encoding, delimiter, duplicate header).
    - Membaca raw dataset.

2. Standarisasi & Validasi Struktur Data
    - Menstandarkan nama kolom ke lowercase.
    - Memvalidasi schema dan data type terhadap satu *schema contract* tunggal.
    - Memvalidasi primary key (`passengerid`).
    - Memvalidasi duplicate record (full-row **dan** business-key).

3. Validasi Kualitas & Domain Data
    - Membuat profiling missing values & memvalidasi threshold missingness.
    - Memvalidasi kualitas string (empty / whitespace) dan format nama.
    - Memvalidasi domain numerik dan domain kategorikal.
    - Memvalidasi hubungan train/test (ID overlap) dan kewajaran jumlah baris.

4. Penyimpanan & Verifikasi Artefak (Staging)
    - Menyimpan dataset ke staged Parquet.
    - Memvalidasi ulang staged artifact (read-back + schema fingerprint match).

5. Dokumentasi, Metadata & Provenance
    - Membuat validation report (JSON + Markdown) yang **selalu tertulis**, baik proses lolos maupun gagal.
    - Membuat ingestion metadata lengkap.
    - Mencatat provenance (git) dan environment.

6. Pembersihan Sistem
    - Membersihkan resource setelah proses selesai.

# Boundary

Notebook ini **tidak melakukan data cleaning substantif**.

Tidak dilakukan:

- imputasi missing value
- penghapusan duplicate
- penghapusan outlier
- encoding
- feature engineering
- perubahan nilai data
- koreksi data source

Normalisasi nama kolom ke lowercase hanya merupakan standardisasi schema teknis. Jika ditemukan masalah pada raw dataset, masalah tersebut **dideteksi dan dilaporkan**, bukan diperbaiki di notebook ini.

# Preparation & Verification

## Import & Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import time
import sys
import csv
import gc
import hashlib
import io
import json
import platform
import subprocess

import polars as pl

# Polars Config
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

# Path Project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# print(PROJECT_ROOT)

## Path Configuration

In [ ]:
# Raw location
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_RAW = RAW_DIR / "train.csv"
TEST_RAW = RAW_DIR / "test.csv"

# Staging location
STAGED_DIR = PROJECT_ROOT / "data" / "stage"

# Parquet
TRAIN_PARQUET = STAGED_DIR / "train.parquet"
TEST_PARQUET = STAGED_DIR / "test.parquet"

# Report
METADATA_DIR = STAGED_DIR / "metadata"
HISTORY_DIR = METADATA_DIR / "history"

INGESTION_METADATA = METADATA_DIR / "ingestion_metadata.json"
VALIDATION_REPORT = METADATA_DIR / "validation_report.json"

VALIDATION_REPORT_MD = METADATA_DIR / "validation_report.md"

# Dir
STAGED_DIR.mkdir(parents = True, exist_ok = True)
METADATA_DIR.mkdir(parents = True, exist_ok = True)
HISTORY_DIR.mkdir(parents = True, exist_ok = True)

print(f"TRAIN_RAW  : {TRAIN_RAW}")
print(f"TEST_RAW   : {TEST_RAW}")
print(f"STAGED_DIR : {STAGED_DIR}")
print(f"METADATA   : {METADATA_DIR}")
print(f"HISTORY    : {HISTORY_DIR}")

## Contract

In [ ]:
DATASET_NAME = "titanic"
PIPELINE_STAGE = "ingestion"
PIPELINE_VERSION = "1.0.0"
SCHEMA_VERSION = "1.0.0"
DQ_RULES_VERSION = "1.0.0"
SOURCE_FORMAT = "csv"
TARGET_FORMAT = "parquet"

print(f"Dataset name        : {DATASET_NAME}")
print(f"Pipeline stage      : {PIPELINE_STAGE}")
print(f"Pipeline version    : {PIPELINE_VERSION}")
print(f"Schema version      : {SCHEMA_VERSION}")
print(f"DQ rules version    : {DQ_RULES_VERSION} \n")

print(f"Source format       : {SOURCE_FORMAT}")
print(f"Target format       : {TARGET_FORMAT}")


## Run Identification

In [ ]:
# Waktu awal
INGESTION_START = time.perf_counter()

# Timezone
WIB = timezone(timedelta(hours=7))
RUN_TIMESTAMP = datetime.now(WIB)

# Format custom
STARTED_FORMATTED = RUN_TIMESTAMP.strftime("%Y-%m-%d %H:%M:%S WIB")

RUN_ID = f"{DATASET_NAME}-{PIPELINE_STAGE}-{STARTED_FORMATTED}"


print(f"Run ID  : {RUN_ID}")
print(f"Started : {STARTED_FORMATTED}")

# Schema Contract

## Blueprint Data - Sudah cek manual (nama kolom, data type, dsb)

| Variable      | Definition                | Key                   | Role          | (Required, Null)  | Reason                                      |
| ------------- | ------------------------- | --------------------- | ------------- | ----------------- | ------------------------------------------- |
| PassengerId   | Table Identifier          |                       | Primary Key   | (True, False)     | Unique ID data                              |
| Survived      | Survival                  | `0` = No, `1` = Yes   | Target        | (True, False)     | Target result                               |
| Pclass        | Ticket Class              | `1` = 1st (Upper),    | Category      | (True, False)     | Customer category                           |
|               |                           | `2` = 2nd (Middle),   |               |                   |                                             |
|               |                           | `3` = 3rd (Lower)     |               |                   |                                             |
| Name          | Passenger Name            |                       | Text          | (True, False)     | Personal customers data                     |
| Sex           | Sex                       |                       | Category      | (True, False)     | Personal customers data                     |
| Age           | Age in years              |                       | Numeric       | (False, True)     | Null data                                   |
| SibSp         | siblings / spouses aboard |                       | Numeric       | (True, False)     | Feature engineering (for travel group size) |
| Parch         | parents / children aboard |                       | Numeric       | (True, False)     | Feature engineering (for travel group size) |
| Ticket        | Ticket number             |                       | Text          | (True, False)     | Customer must have ticket                   |
| Fare          | Passenger Fee             |                       | Numeric       | (False, True)     | Null data                                   |
| Cabin         | Cabin number              |                       | Text          | (False, True)     | Null data                                   |
| Embarked      | Port of Embarkation       | `C` = Cherbourg,      | Category      | (False, True)     | Null data                                   |
|               |                           | `Q` = Queenstown,     |               |                   |                                             |
|               |                           | `S` = Southampton     |               |                   |                                             |

## Required

In [ ]:
REQUIRED_SCHEMA_CONTRACT = {
    "passengerid": {
        "dtype": "Int64",
        "semantic_type": "primary_key",
    },
    "survived": {
        "dtype": "Int64",
        "semantic_type": "binary_target",
        "allowed_values": [0, 1],
    },
    "pclass": {
        "dtype": "Int64",
        "semantic_type": "categorical",
        "allowed_values": [1, 2, 3],
    },
    "name": {
        "dtype": "String",
        "semantic_type": "text",
    },
    "sex": {
        "dtype": "String",
        "semantic_type": "categorical",
        "allowed_values": ["male", "female"],
    },
    "sibsp": {
        "dtype": "Int64",
        "semantic_type": "count",
        "min": 0,
    },
    "parch": {
        "dtype": "Int64",
        "semantic_type": "count",
        "min": 0,
    },
    "ticket": {
        "dtype": "String",
        "semantic_type": "identifier",
    },
}

### Other

In [ ]:
OTHER_SCHEMA_CONTRACT = {
    "age": {
        "dtype": "Float64",
        "semantic_type": "numeric",
        "min": 0,
        "max": 100,
        "max_null_ratio": 0.3,
    },
    "fare": {
        "dtype": "Float64",
        "semantic_type": "numeric",
        "min": 0,
        "max_null_ratio": 0.05,
    },
    "cabin": {
        "dtype": "String",
        "semantic_type": "categorical_text",
        "max_null_ratio": 0.9,
    },
    "embarked": {
        "dtype": "String",
        "semantic_type": "categorical",
        "allowed_values": ["C", "Q", "S"],
        "max_null_ratio": 0.05,
    },
}

In [ ]:
# 
SCHEMA_CONTRACT = {}

# filter for required
for col, spec in REQUIRED_SCHEMA_CONTRACT.items():
    rule = spec.copy()
    rule["nullable"] = False
    rule["required"] = True
    SCHEMA_CONTRACT[col] = rule

# filter for other
for col, spec in OTHER_SCHEMA_CONTRACT.items():
    rule = spec.copy()
    rule["nullable"] = True
    rule["required"] = False
    SCHEMA_CONTRACT[col] = rule

In [ ]:
print(f"✓ REQUIRED_SCHEMA_CONTRACT : {len(REQUIRED_SCHEMA_CONTRACT)} kolom")
print(f"✓ OTHER_SCHEMA_CONTRACT    : {len(OTHER_SCHEMA_CONTRACT)} kolom \n")

print(f"✓ Total SCHEMA_CONTRACT    : {len(SCHEMA_CONTRACT)} kolom \n")
print(f"--- ISI SCHEMA_CONTRACT ---")
print(json.dumps(SCHEMA_CONTRACT, indent = 4))

### Contract Filter

In [ ]:
# Default for most data
DEFAULT_TYPE = {
    "Int64": pl.Int64,
    "Float64": pl.Float64,
    "String": pl.String,
}

# List Shcema Order
COLUMN_ORDER = list(SCHEMA_CONTRACT.keys())

print(f"Urutan di Schema: {COLUMN_ORDER}")

In [ ]:
EXPECTED_DTYPES = {
    column: DEFAULT_TYPE[spec["dtype"]]
    for column, spec in SCHEMA_CONTRACT.items()
}

STRING_COLUMNS = [
    column for column, spec in SCHEMA_CONTRACT.items() if spec["dtype"] == "String"
]

NUMERIC_BOUNDS = {
    column: {"min": spec.get("min"), "max": spec.get("max")}
    for column, spec in SCHEMA_CONTRACT.items()
    if spec["dtype"] in ("Int64", "Float64") and ("min" in spec or "max" in spec)
}

ALLOWED_VALUES = {
    column: set(spec["allowed_values"])
    for column, spec in SCHEMA_CONTRACT.items()
    if "allowed_values" in spec
}

print(f"Tipe data: {EXPECTED_DTYPES}\n")
print(f"String coloumn: {STRING_COLUMNS}\n")
print(f"Numerical coloumn: {NUMERIC_BOUNDS}\n")
print(f"Only allowed values: {ALLOWED_VALUES}\n")

In [ ]:
# 
TRAIN_DIFF_COLS = {"survived"}

EXPECTED_TRAIN_BP = COLUMN_ORDER

EXPECTED_TEST_BP = [
    column for column in COLUMN_ORDER if column not in TRAIN_DIFF_COLS
]

print(f"Ekspektasi column train: \n{EXPECTED_TRAIN_BP}\n")
print(f"Ekspektasi column test: \n{EXPECTED_TEST_BP}")

In [ ]:
MISSINGNESS_THRESHOLDS = {
    column: spec.get("max_null_ratio", 0.0 if not spec["nullable"] else 1.0)
    for column, spec in SCHEMA_CONTRACT.items()
}

REQUIRED_NON_NULL = {
    "train": [
        column
        for column in EXPECTED_TRAIN_BP
        if not SCHEMA_CONTRACT[column]["nullable"]
    ],
    
    "test": [
        column
        for column in EXPECTED_TEST_BP
        if not SCHEMA_CONTRACT[column]["nullable"]
    ],
}

print(f"MISSINGNESS_THRESHOLDS: \n{MISSINGNESS_THRESHOLDS} \n")

print(f"REQUIRED_NON_NULL[train]: \n{REQUIRED_NON_NULL["train"]} \n")
print(f"REQUIRED_NON_NULL[test]: \n{REQUIRED_NON_NULL["test"]} \n")

# Validation Rule

In [ ]:
#
MISSING_VALUES = ["", "NA", "N/A", "na", "n/a", "N/a"]

## Integrity

In [ ]:
from src.ingestion.source_file import check_source_file
from src.ingestion.source_fingerprint import check_source_fingerprint

### Source Validation

In [ ]:
# Pengecekan file
train_source_check = check_source_file(file_path = TRAIN_RAW, dataset_name = "train")

In [ ]:
# Pengecekan file
test_source_check = check_source_file(file_path = TEST_RAW, dataset_name = "test")

### Source Fingerprint

In [ ]:
train_fingerprint_check = check_source_fingerprint(file_path = TRAIN_RAW, dataset_name = "train")

In [ ]:
test_fingerprint_check = check_source_fingerprint(file_path = TEST_RAW, dataset_name = "test")